# 🛒 Telegram бот — Сидорович из S.T.A.L.K.E.R.
Запускай ячейки по порядку сверху вниз.

## Шаг 1 — Установка зависимостей
Запусти один раз, потом можно пропускать.

In [1]:
!pip install python-telegram-bot==21.6 httpx nest_asyncio python-dotenv -q
print('Зависимости установлены ✓')

Зависимости установлены ✓



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Шаг 2 — Настройки
Токен берётся из файла `.env` в папке проекта.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

TELEGRAM_TOKEN  = os.getenv('TELEGRAM_TOKEN')
LM_STUDIO_URL   = 'http://localhost:1234/v1/chat/completions'
MODEL_NAME      = 'qwen/qwen3.5-9b'   # точное имя из LM Studio
MAX_HISTORY     = 20
TEMPERATURE     = 0.8
MAX_TOKENS      = 2048
REQUEST_TIMEOUT = 120.0
MSG_DELAY       = 0.3

if not TELEGRAM_TOKEN:
    raise ValueError('TELEGRAM_TOKEN не найден! Проверь файл .env')

print('Настройки загружены ✓')

Настройки загружены ✓


## Шаг 3 — Системный промпт

In [3]:
SYSTEM_PROMPT = """Ты — Сидорович, торговец из игры S.T.A.L.K.E.R. Ты сидишь в своём бункере в Кордоне и торгуешь с сталкерами — покупаешь артефакты, продаёшь снаряжение, даёшь задания.

Твой характер:
- Жадный, хитрый, себе на уме торгаш средних лет
- Всегда думаешь о выгоде — своей прежде всего
- Говоришь покровительственно, немного свысока, но вежливо если это выгодно
- Знаешь всех и всё что происходит в Зоне — информация это тоже товар
- Любишь деньги и артефакты больше всего на свете
- Можешь войти в доверие, но предашь если цена будет хорошей
- Иногда даёшь задания сталкерам в обмен на снаряжение или деньги
- Никогда не выходишь из роли. Ты НЕ языковая модель, ты Сидорович.
- Отвечай коротко и по делу, без лишних размышлений.

Примеры твоей речи:
- 'А, снова ты. Ну что принёс на этот раз?'
- 'Хм, интересный артефакт... но рынок сейчас упал, больше трёхсот не дам.'
- 'Есть одно дельце... не для слабонервных, зато заплачу хорошо.'
- 'Информация стоит денег, сталкер. Ничего бесплатного в Зоне нет.'
- 'Ты жив — уже хорошо. Значит, снова можем поторговать.'"""

print('Промпт загружен ✓')

Промпт загружен ✓


## Шаг 4 — Импорты

In [4]:
import logging
import asyncio
import httpx
import nest_asyncio
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
from telegram.constants import ChatAction

nest_asyncio.apply()

conversation_history: dict[int, list[dict]] = {}

logging.basicConfig(
    format='%(asctime)s | %(levelname)s | %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)
print('Импорты загружены ✓')

Импорты загружены ✓


## Шаг 5 — Дополнительные функции (поиск, погода, новости, задания, валюта)

In [5]:
# ══════════════════════════════════════════════
# ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ СИДОРОВИЧА
# ══════════════════════════════════════════════
import random

# --- Поиск в интернете (DuckDuckGo, без API ключа) ---
async def web_search(query: str) -> str:
    try:
        url = f'https://api.duckduckgo.com/?q={query}&format=json&no_html=1&skip_disambig=1'
        async with httpx.AsyncClient(timeout=10.0) as client:
            r = await client.get(url, headers={'User-Agent': 'Mozilla/5.0'})
            data = r.json()
            result = data.get('AbstractText') or data.get('Answer') or ''
            if not result and data.get('RelatedTopics'):
                result = data['RelatedTopics'][0].get('Text', '')
            return result[:500] if result else None
    except Exception:
        return None

# --- Погода (wttr.in, без API ключа) ---
async def get_weather(city: str) -> str:
    try:
        url = f'https://wttr.in/{city}?format=3&lang=ru'
        async with httpx.AsyncClient(timeout=10.0) as client:
            r = await client.get(url, headers={'User-Agent': 'curl/7.68.0'})
            return r.text.strip()
    except Exception:
        return None

# --- Новости (RSS через rss2json) ---
async def get_news() -> list[str]:
    try:
        url = 'https://api.rss2json.com/v1/api.json?rss_url=https://lenta.ru/rss/news'
        async with httpx.AsyncClient(timeout=10.0) as client:
            r = await client.get(url)
            data = r.json()
            items = data.get('items', [])[:5]
            return [item['title'] for item in items]
    except Exception:
        return []

# --- Генератор заданий ---
QUESTS = [
    'Принеси мне артефакт «Огненный шар» из Тёмной долины. Платить буду хорошо — 5000 рублей.',
    'Есть один тип на Свалке, должен мне крупную сумму. Найди его и... напомни об этом. Подробности при встрече.',
    'Военные усилили патрули у Кордона. Нужно выяснить — почему. Разведай и доложи.',
    'Пропал один мой курьер где-то у Агропрома. С ним был важный груз. Найди хоть что-нибудь.',
    'Мне нужны данные с заброшенного блокпоста. ПДА лежит там уже месяц — сходи, забери.',
    'Слышал про новую аномалию у Свалки? Нужен образец артефакта оттуда. Никто не соглашается идти.',
    'Бандиты на Кордоне совсем обнаглели — перекрыли мой торговый путь. Реши этот вопрос как сможешь.',
    'Нужна псевдособачья шкура — штук пять минимум. Есть покупатель на большой земле, платит хорошо.',
    'Зомбированный сталкер бродит у Кордона и пугает новичков. Это плохо для бизнеса. Разберись.',
    'Принеси мне флешку из лаборатории Х-18. Говорят, там интересные данные. Заплачу щедро.',
]

def get_random_quest() -> str:
    return random.choice(QUESTS)

# --- Курс валют (ЦБ РФ) ---
async def get_currency() -> str:
    try:
        url = 'https://www.cbr-xml-daily.ru/daily_json.js'
        async with httpx.AsyncClient(timeout=10.0) as client:
            r = await client.get(url)
            data = r.json()
            usd = data['Valute']['USD']['Value']
            eur = data['Valute']['EUR']['Value']
            tjs = round(usd / 10.9, 2)  # сомони через доллар
            return f'USD: {usd:.2f} руб | EUR: {eur:.2f} руб | 1 USD ≈ {10.9} TJS (сомони)'
    except Exception:
        return None

# --- Определяем намерение пользователя ---
def detect_intent(text: str) -> tuple[str, str]:
    t = text.lower().strip()

    # Погода
    weather_kw = ['погода', 'температура', 'дождь', 'снег', 'ветер', 'прогноз']
    if any(w in t for w in weather_kw):
        stop_words = set(weather_kw + ['в', 'во', 'какая', 'какой', 'сегодня', 'завтра', 'сейчас', 'там', 'какие'])
        words = [w for w in text.split() if w.lower() not in stop_words]
        city = words[0] if words else 'Москва'
        return 'weather', city

    # Новости
    news_kw = ['новости', 'что случилось', 'что происходит', 'последние события']
    if any(w in t for w in news_kw):
        return 'news', ''

    # Задание
    quest_kw = ['задание', 'есть работа', 'есть дело', 'дай задание', 'что за работа']
    if any(w in t for w in quest_kw):
        return 'quest', ''

    # Курс валют
    currency_kw = ['курс', 'доллар', 'евро', 'валюта', 'рубль', 'сомони']
    if any(w in t for w in currency_kw):
        return 'currency', ''

    # Поиск
    search_kw = ['что такое', 'кто такой', 'что это', 'найди', 'расскажи про', 'информация о', 'где находится']
    if any(w in t for w in search_kw):
        return 'search', text

    return 'chat', ''

# --- Обёртка ответа в стиль Сидоровича ---
async def sidorovich_wrap(intent: str, data: str, original_text: str) -> str:
    if intent == 'weather':
        prompt = f'Ты Сидорович из STALKER. Скажи сталкеру про погоду в своём стиле. Данные: {data}. Коротко, 1-2 предложения.'
    elif intent == 'news':
        prompt = f'Ты Сидорович из STALKER. Передай эти новости как слухи с большой земли. Данные: {data}. В своём стиле, кратко.'
    elif intent == 'search':
        prompt = f'Ты Сидорович из STALKER. Сталкер спросил: "{original_text}". Ответь используя эту информацию: {data}. В своём стиле торговца.'
    elif intent == 'currency':
        prompt = f'Ты Сидорович из STALKER. Прокомментируй курс валют как торговец. Данные: {data}. Коротко.'
    else:
        return data

    messages = [{'role': 'user', 'content': prompt}]
    payload = {'model': MODEL_NAME, 'messages': messages, 'temperature': 0.8,
               'max_tokens': 200, 'stream': False, 'enable_thinking': False}
    try:
        async with httpx.AsyncClient(timeout=30.0) as client:
            r = await client.post(LM_STUDIO_URL, json=payload)
            msg = r.json()['choices'][0]['message']
            text = (msg.get('content') or '').strip()
            if not text:
                reasoning = (msg.get('reasoning_content') or '')
                lines = [l.strip() for l in reasoning.split('\n') if l.strip()]
                for line in reversed(lines):
                    if len(line) > 15:
                        text = line.strip('"').strip("'")
                        break
            return text or data
    except Exception:
        return data

print('Дополнительные функции загружены ✓')

Дополнительные функции загружены ✓


## Шаг 6 — Логика бота

In [6]:
async def get_llm_response(messages: list) -> str:
    payload = {
        'model': MODEL_NAME,
        'messages': messages,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'stream': False,
        'enable_thinking': False,
    }
    try:
        async with httpx.AsyncClient(timeout=REQUEST_TIMEOUT) as client:
            response = await client.post(LM_STUDIO_URL, json=payload)
            response.raise_for_status()
            data = response.json()
            message = data['choices'][0]['message']
            text = (message.get('content') or '').strip()
            if not text:
                reasoning = (message.get('reasoning_content') or '').strip()
                for marker in ['Selected Response', 'Final Polish', 'settle on', "Let's go with"]:
                    if marker in reasoning:
                        after = reasoning.split(marker)[-1]
                        lines = [l.strip().strip('"').strip("'") for l in after.split('\n') if l.strip()]
                        if lines:
                            text = lines[0]
                            break
                if not text:
                    lines = [l.strip() for l in reasoning.split('\n') if l.strip()]
                    for line in reversed(lines):
                        clean = line.strip('"').strip("'").strip()
                        if len(clean) > 10:
                            text = clean
                            break
            return text or '...помехи на связи. Попробуй ещё раз, сталкер.'
    except httpx.ConnectError:
        return '⚠️ LM Studio не отвечает. Запущен сервер?'
    except httpx.TimeoutException:
        return '⏱ Модель думает слишком долго. Попробуй ещё раз.'
    except Exception as e:
        logger.error(f'Ошибка запроса: {e}')
        return '❌ Что-то пошло не так. Попробуй ещё раз.'


async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    conversation_history[user_id] = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    await update.message.reply_text(
        'А, здравствуй. Заходи, не стой на пороге. Что принёс? Или пришёл с пустыми руками?\n\n'
        '/start — начать заново\n'
        '/clear — забыть всё\n'
        '/quest — получить задание\n'
        '/news — новости с большой земли\n'
        '/currency — курс валют'
    )


async def cmd_clear(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    conversation_history[user_id] = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    await update.message.reply_text('Ладно, забыли. Начинаем с чистого листа.')


async def cmd_quest(update: Update, context: ContextTypes.DEFAULT_TYPE):
    quest = get_random_quest()
    await update.message.reply_text(f'Слушай внимательно, сталкер...\n\n{quest}')


async def cmd_news(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await context.bot.send_chat_action(chat_id=update.effective_chat.id, action=ChatAction.TYPING)
    headlines = await get_news()
    if headlines:
        text = 'Слухи с большой земли:\n\n' + '\n'.join(f'• {h}' for h in headlines)
    else:
        text = 'Связи нет... радио молчит. Попробуй позже.'
    await update.message.reply_text(text)


async def cmd_currency(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await context.bot.send_chat_action(chat_id=update.effective_chat.id, action=ChatAction.TYPING)
    data = await get_currency()
    if data:
        result = await sidorovich_wrap('currency', data, '')
        await update.message.reply_text(result)
    else:
        await update.message.reply_text('Биржа не отвечает. Бывает.')


async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.id
    user_text = (update.message.text or '').strip()
    if not user_text:
        return

    if user_id not in conversation_history:
        conversation_history[user_id] = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    await context.bot.send_chat_action(chat_id=update.effective_chat.id, action=ChatAction.TYPING)

    intent, query = detect_intent(user_text)

    if intent == 'weather':
        data = await get_weather(query)
        response_text = await sidorovich_wrap('weather', data or 'данные недоступны', user_text) if data else 'Метеостанция молчит, сталкер.'

    elif intent == 'news':
        headlines = await get_news()
        if headlines:
            data = ', '.join(headlines)
            response_text = await sidorovich_wrap('news', data, user_text)
        else:
            response_text = 'Связи нет... радио молчит.'

    elif intent == 'quest':
        response_text = 'Слушай внимательно...\n\n' + get_random_quest()

    elif intent == 'currency':
        data = await get_currency()
        response_text = await sidorovich_wrap('currency', data, user_text) if data else 'Биржа не отвечает.'

    elif intent == 'search':
        data = await web_search(query)
        if data:
            response_text = await sidorovich_wrap('search', data, user_text)
        else:
            # Если поиск не дал результата — обычный чат
            conversation_history[user_id].append({'role': 'user', 'content': user_text})
            response_text = await get_llm_response(conversation_history[user_id])
            conversation_history[user_id].append({'role': 'assistant', 'content': response_text})
            await update.message.reply_text(response_text)
            return

    else:  # обычный чат
        conversation_history[user_id].append({'role': 'user', 'content': user_text})
        if len(conversation_history[user_id]) > MAX_HISTORY:
            conversation_history[user_id] = [conversation_history[user_id][0]] + conversation_history[user_id][-(MAX_HISTORY-1):]
        response_text = await get_llm_response(conversation_history[user_id])
        conversation_history[user_id].append({'role': 'assistant', 'content': response_text})

    await update.message.reply_text(response_text)


print('Логика загружена ✓')

Логика загружена ✓


## Шаг 6 — Запуск бота
> ⚠️ Убедись что LM Studio запущен!

Для остановки — **Interrupt (■)**, потом **Restart kernel (⟳)**.

In [7]:
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)

app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
app.add_handler(CommandHandler('start', cmd_start))
app.add_handler(CommandHandler('clear', cmd_clear))
app.add_handler(CommandHandler('quest', cmd_quest))
app.add_handler(CommandHandler('news', cmd_news))
app.add_handler(CommandHandler('currency', cmd_currency))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

print('✅ Бот запущен. Сидорович в бункере 🛒')
print('Для остановки — Interrupt (■), потом Restart kernel (⟳)')

app.run_polling(drop_pending_updates=True)

✅ Бот запущен. Сидорович в бункере 🛒
Для остановки — Interrupt (■), потом Restart kernel (⟳)


2026-05-21 14:48:11,689 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/getMe "HTTP/1.1 200 OK"
2026-05-21 14:48:11,867 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/deleteWebhook "HTTP/1.1 200 OK"
2026-05-21 14:48:11,869 | INFO | Application started
2026-05-21 14:48:17,204 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/getUpdates "HTTP/1.1 200 OK"
2026-05-21 14:48:17,696 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/sendChatAction "HTTP/1.1 200 OK"
2026-05-21 14:48:18,864 | INFO | HTTP Request: GET https://api.rss2json.com/v1/api.json?rss_url=https://lenta.ru/rss/news "HTTP/1.1 200 OK"
2026-05-21 14:48:19,090 | INFO | HTTP Request: POST https://api.telegram.org/bot8697925205:AAET64g_C3u1-Q53omGBcZfFms3RbHl3qQ0/sendMessage "HTTP/1.1 200 OK"
2026-05-21 14:48:27,366 

RuntimeError: Cannot close a running event loop